# Lab 18: Spark MLlib

**Module 18** | Read `notes/18-spark-mllib.pdf` before running this notebook.

Build a Spark ML pipeline on the credit-fraud sample and compare fit time with scikit-learn on the same rows.

Run every cell top to bottom. Code is complete, no fill-in sections. Markdown cells explain what each block does and why.


## Runtime device

PyTorch can run on your NVIDIA GPU through CUDA or fall back to CPU. GPU execution moves matrix operations off the host and typically speeds up neural network training by a large factor. The next cell detects what is available and prints the active device so you know whether to expect fast training or should keep batch sizes small.


In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"CUDA enabled, {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")
else:
    print("CUDA not available, using CPU. Labs still run; training may take longer.")


## Distributed ML with PySpark MLlib

Credit-card fraud detection is a classic **imbalanced classification** problem.
The sample dataset contains anonymized features **V1 to V28** (PCA components) plus `Amount`, `Time`, and a binary `Class` label.

Spark MLlib runs pipelines across a cluster (here: `local[1]` on one machine) using DataFrame operations.
We build a **Pipeline** with `VectorAssembler` then `RandomForestClassifier`, then compare fit time and accuracy against **scikit-learn** on the same rows.

If `credit_fraud_sample.parquet` is missing, the lab falls back to the CSV version automatically.


### Load data and inspect schema

Before training, confirm column names, preview a few rows, and check the fraud vs legitimate class balance.


In [ ]:
import sys
from pathlib import Path

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from runtime_env import configure_runtime, create_spark, stop_spark

configure_runtime(quiet=True)

from pyspark.sql import functions as F

parquet_path = ROOT / "datasets" / "credit_fraud_sample.parquet"
csv_path = ROOT / "datasets" / "credit_fraud_sample.csv"
feature_cols = [f"V{i}" for i in range(1, 29)]

spark = create_spark("Lab18")
if parquet_path.exists():
    df = spark.read.parquet(str(parquet_path))
    source_name = parquet_path.name
else:
    df = spark.read.option("header", True).option("inferSchema", True).csv(str(csv_path))
    source_name = csv_path.name

row_count = df.count()
print(f"Loaded {source_name}: {row_count:,} rows")
print()
print("Schema:")
df.printSchema()
print("Sample rows (first 3):")
df.select("Time", "Amount", "V1", "V2", "V28", "Class").show(3, truncate=False)

balance = df.groupBy("Class").count().orderBy("Class").collect()
print("Class balance:")
for row in balance:
    label = "legit" if row["Class"] == 0 else "fraud"
    pct = 100.0 * row["count"] / row_count
    print(f"  Class {int(row['Class'])} ({label}): {row['count']:,} rows ({pct:.2f}%)")


### Train with Spark MLlib

The pipeline assembles V1 to V28 into a feature vector, fits a 50-tree Random Forest, and evaluates on a held-out 20% split.


In [ ]:
import time

from pyspark.ml import Pipeline
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
rf = RandomForestClassifier(
    labelCol="Class",
    featuresCol="features",
    numTrees=50,
    maxDepth=8,
    seed=42,
)
pipeline = Pipeline(stages=[assembler, rf])

train, test = df.randomSplit([0.8, 0.2], seed=42)
train_count = train.count()
test_count = test.count()
print(f"Train rows: {train_count:,}, test rows: {test_count:,}")

t0 = time.perf_counter()
spark_model = pipeline.fit(train)
spark_fit_sec = time.perf_counter() - t0

predictions = spark_model.transform(test)
evaluator = MulticlassClassificationEvaluator(
    labelCol="Class", predictionCol="prediction", metricName="accuracy"
)
spark_accuracy = evaluator.evaluate(predictions)

print(f"Spark fit time: {spark_fit_sec:.3f} s")
print(f"Spark test accuracy: {spark_accuracy:.4f}")
print()
print("Confusion-style counts (actual Class vs predicted):")
conf_rows = (
    predictions.groupBy("Class", "prediction")
    .count()
    .orderBy("Class", "prediction")
    .collect()
)
print(f"{'Actual':>8} | {'Predicted':>10} | {'Count':>8}")
print("-" * 32)
for row in conf_rows:
    actual = int(row["Class"])
    predicted = int(row["prediction"])
    print(f"{actual:>8} | {predicted:>10} | {row['count']:>8}")

stop_spark(spark)


### Compare with scikit-learn on the same features

We reload the data into pandas, stratify the train/test split, fit an equivalent Random Forest, and print a side-by-side timing and accuracy table.


In [ ]:
import time

import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split

if parquet_path.exists():
    pdf = pd.read_parquet(parquet_path)
else:
    pdf = pd.read_csv(csv_path)

X = pdf[feature_cols]
y = pdf["Class"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Pandas train rows: {len(X_train):,}, test rows: {len(X_test):,}")

t0 = time.perf_counter()
sk_model = RandomForestClassifier(
    n_estimators=50, max_depth=8, random_state=42, n_jobs=-1
)
sk_model.fit(X_train, y_train)
sklearn_fit_sec = time.perf_counter() - t0

sk_preds = sk_model.predict(X_test)
sk_accuracy = accuracy_score(y_test, sk_preds)
cm = confusion_matrix(y_test, sk_preds)

print(f"sklearn fit time: {sklearn_fit_sec:.3f} s")
print(f"sklearn test accuracy: {sk_accuracy:.4f}")
print()
print("sklearn confusion matrix (rows=actual, cols=predicted):")
print(f"              pred 0    pred 1")
print(f"  actual 0    {cm[0, 0]:6d}    {cm[0, 1]:6d}")
print(f"  actual 1    {cm[1, 0]:6d}    {cm[1, 1]:6d}")

print()
print("=" * 55)
print("SPARK vs SKLEARN TIMING TABLE")
print("=" * 55)
print(f"{'Metric':<22} | {'Spark':>12} | {'sklearn':>12}")
print("-" * 55)
print(f"{'Fit time (seconds)':<22} | {spark_fit_sec:>12.3f} | {sklearn_fit_sec:>12.3f}")
print(f"{'Test accuracy':<22} | {spark_accuracy:>12.4f} | {sk_accuracy:>12.4f}")
ratio = spark_fit_sec / sklearn_fit_sec if sklearn_fit_sec > 0 else float("inf")
print(f"{'Fit-time ratio Spark/sklearn':<22} | {ratio:>12.2f}x | {'1.00x':>12}")


## Spark vs scikit-learn

On a **small local sample** (~5k rows), scikit-learn usually **fits faster** because Spark pays JVM startup, DataFrame planning, and serialization overhead.
Spark's advantage appears at **millions+ of rows** on a cluster where data does not fit in memory on one machine.

Accuracies should be **close but not identical**: different random splits (Spark `randomSplit` vs sklearn `train_test_split`) and implementation details cause small differences.

**When to use which:**

- **sklearn:** prototyping, datasets that fit in RAM, single-machine speed.
- **Spark MLlib:** large-scale ETL + training in one pipeline, data already in a lake/warehouse, distributed feature engineering.
